# Synthèse technique détaillée — Écosystème d'IA
**Cours** : Mise en place d'un écosystème d'IA — AEC (Spécialisation technique)
**Enseignant** : Dr. Komi Sodoké
**Date** : 06/06/2026

## Résumé
Document consolidé avec analyse technique approfondie, checklist d'audit, Dockerfile optimisé, script de Watchdog et schéma d'architecture.


## Schéma d'architecture simplifié
Le diagramme ci‑dessous illustre les composants principaux : CPU, PCIe, GPUs, NVMe storage et stack de monitoring (Prometheus/Grafana).
![](images/architecture.png)

## Détails techniques approfondis
### PCIe et goulots détranglement
- PCIe Gen4 x16 fournit théoriquement ~32 GB/s par direction. Dans une configuration multi-GPU sans NVLink, le trafic CPU↔GPU peut saturer le bus si plusieurs GPUs transfèrent simultanément des batches volumineux.
- Pour entraînement distribué intensif, privilégier NVLink (ou NVSwitch) entre GPUs pour réduire les transferts via CPU et PCIe.

### Ratio CPU/GPU recommandé
- Pour charge d'entraînement classique, viser 8–16 CPU threads par GPU pour la préparation des lots (data loading, augmentation). Pour charges heavy I/O (préprocessing intensif), augmenter les cœurs ou utiliser des workers dédiés (DALI, DataLoader num_workers).

### Taille de VRAM et stratégie de sharding
- Si modèle < VRAM unique (ex. 40 GB), préférer single-GPU training pour éviter la complexité.
- Pour modèles > VRAM GPU, utiliser ZeRO / model parallelism ou pipeline parallelism. ZeRO stage 2/3 réduit l'empreinte mémoire mais impose coûts réseau/communication.

## Commandes pratiques — vérification & installation
### Vérifier GPU et driver
```
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader,nounits
```
### Installer PyTorch compatible (exemple pour CUDA 12.1)
```
pip3 install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio --extra-index-url https://pypi.org/simple
```
Remarque : Adapter l'URL/cuXX selon la version CUDA affichée par `nvidia-smi`.

## Dockerfile recommandé (optimisé)
Explication : on utilise l'image runtime CUDA et on installe la roue PyTorch adaptée à la version CUDA pour éviter les incompatibilités.
```dockerfile
FROM nvidia/cuda:12.1.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y \
    python3-pip \
    python3-dev \
    && rm -rf /var/lib/apt/lists/*
# Installer pip wheel PyTorch pour CUDA 12.1
RUN pip3 install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio pandas
WORKDIR /app
COPY train.py /app/
CMD ["python3", "train.py"]
```
Conseil : construisez l'image via CI en testant l'exécution d'un petit script GPU afin de valider la compatibilité avant déploiement.

## Monitoring — architecture et métriques clés
- Collecte : exporter métriques via node_exporter (CPU, mémoire, disque) et exporter GPU metrics via `nvidia-dcgm-exporter` ou `prometheus-nvidia-smi-exporter`.
- Visualisation : Grafana dashboards (GPU temperature, power draw, GPU utilization, memory usage, ECC errors).
- Alerting : Alertmanager pour règles (temp >82°C for 60s -> page on-call; ECC errors >10 -> isolate node).

In [ ]:
# Exemple : script d'extraction nvidia-smi vers Prometheus textfile (extrait)
import subprocess
out = subprocess.check_output(['nvidia-smi','--query-gpu=name,temperature.gpu,memory.total','--format=csv,noheader,nounits'])
print(out.decode())

## Annexes
- Annexe A : Dockerfile complet (voir précédent notebook).
- Annexe B : Script Watchdog (fichier `scripts/watchdog_simulate.py`).
- Annexe C : Output simulation (fichier `scripts/watchdog_output.txt`).
- Annexe D : Document Word généré (`Synthese_Ecosysteme_IA.docx`).